# 第五次课后练习


<span style="color:red; font-weight:bold;">请将作业文件命名为 第五次课后练习+姓名+学号.ipynb, 例如 第五次课后练习+张三+1000000000.ipynb</span>

<span style="color:red; font-weight:bold;">在作业过程中觉得有心得或者自己拓展学习到有价值内容的，可以在文件名最后加一个#号。例如第五次课后练习+张三+1000000000+#.ipynb</span>



# 第零部分 代码理解

请认真阅读代码，理解代码的功能，先写出预想的结果。运行并检验结果是否如预期。如果不如预期，请分析理解其中的原因

由于 Jupyter Notebook 无法直接运行多进程程序，我们采用一种迂回的方法：使用`%%writefile`将要运行的程序写入到一个`.py`文件中，再获取运行该文件所得到的结果

### 0.0 在notebook中尝试运行多进程

In [1]:
%%writefile multiprocessing_script.py
from multiprocessing import Pool
import time
import random

def square(x):
    time.sleep(1)
    return x*x

if __name__ == "__main__":
    
    with Pool(5) as p:
        start_time = time.time()
        result = p.map(square, range(20))
        print("Result:", result)
        end_time = time.time()
    print(f"All done, time taken: {end_time - start_time:.2f}")

Writing multiprocessing_script.py


# 超清晰总结：
**Pool 进程池自带一整套「自动分配任务」的函数**，全都不用你手动管进程，它们**自动把任务分给 Pool(n) 里的 n 个进程**。


---

# 一、Pool 里**自动分配多进程**的 4 个核心函数
全部都是：**你只管扔任务 → 系统自动分给进程池里的进程跑**

## 1. **map()** —— 最常用、最简单
- 按顺序执行
- 阻塞（等全部做完才返回）
- 函数只能传 **1 个参数**
```python
p.map(square, [1,2,3,4])
```

## 2. **starmap()** —— 多参数版本
和 map 一模一样，**但支持函数多个参数**
```python
def add(a, b):
    return a + b

p.starmap(add, [(1,2), (3,4), (5,6)])
```

## 3. **map_async()** —— 异步非阻塞
不用等任务跑完，程序可以继续做别的
```python
result = p.map_async(square, [1,2,3])
print("我不用等！先执行别的")
print(result.get())  # 最后再拿结果
```

## 4. **apply_async()** —— 单次异步任务
只提交**一个函数调用**，但依然自动进进程池
```python
p.apply_async(square, args=(5,))
```

---

# 二、你问的重点：
## **我想对一个函数分配进程，应该怎么写？**

不用手动分配！
**Pool 会自动、智能、均匀分配任务**

你只需要写 2 行：

### 1. 创建池子（固定 n 个进程）
```python
pool = Pool(5)  # 5个进程永远在这
```

### 2. 扔函数 + 任务进去
池子自动分配！

---

# 三、最标准、最实用的写法（直接复制用）
## 写法 1：单参数函数 → map
```python
def task(x):
    time.sleep(1)
    return x*x

if __name__ == "__main__":
    with Pool(5) as p:
        results = p.map(task, [0,1,2,3,4,5,6,7,8,9])
```

## 写法 2：多参数函数 → starmap（超级常用）
```python
def task(a, b, c):
    time.sleep(1)
    return a + b + c

if __name__ == "__main__":
    with Pool(5) as p:
        results = p.starmap(task, [(1,2,3), (4,5,6), (7,8,9)])
```

## 写法 3：不想等、异步执行 → map_async
```python
with Pool(5) as p:
    res = p.map_async(task, range(10))
    print("任务还在跑，我先干别的")
    print(res.get())  # 拿结果
```

---

# 四、最重要的一句话（必记）
## **Pool(n) 创建 n 个进程
## 所有 map / starmap / apply_async / map_async
## 都会自动把任务分配给这 n 个进程！**

你**完全不用手动分配哪个进程跑哪个任务**
系统自动调度、自动负载均衡。

---

# 五、你最关心的问题答案：
### Q：我想让我的函数自动使用多进程，怎么写？
### A：
1. 定义你的函数
2. 创建 Pool(进程数)
3. 用 **map / starmap** 扔进去
4. 完事！自动并行！

---

### 总结
- **map**：单参数、顺序、阻塞
- **starmap**：多参数、顺序、阻塞
- **map_async**：异步、不阻塞
- **apply_async**：单个任务异步

全部**自动分配进程池里的进程**，不用你管。

---

如果你愿意，我可以**直接帮你把你的业务函数改成多进程版本**！
你只要把你的函数发我就行～

In [2]:
import subprocess  
# subprocess 模块允许你在Python程序中创建新的进程，
# 并获取它们的输入/输出/错误输出以及返回码。capture_output=True

result = subprocess.run(
    ["python", "multiprocessing_script.py"], 
    capture_output=True, text=True)
print(result.stdout)

我不用等！先执行别的
[1, 4, 9]
Result: [0, 1, 4, 9, 16, 25, 36, 49, 64, 81, 100, 121, 144, 169, 196, 225, 256, 289, 324, 361]
All done, time taken: 5.19



subprocess 建立了一个新的进程来运行命令，并返回一个子进程对象。capture_output参数可以捕获子进程的输出，并返回一个包含标准输出和标准错误的字节字符串。text参数可以将字节字符串转换为字符串。

## **0.1** 多进程编程，进程池，进程间通讯
    阅读下面代码，观察四次运行的结果，解释出现这个结果的原因。

In [3]:
%%writefile multiprocessing_script.py
import multiprocessing
import time
from multiprocessing import Pool, Manager

class CustomWorker:
    def __init__(self, worker_id, task_queue, result_queue, config):
        self.worker_id = worker_id
        self.task_queue = task_queue
        self.result_queue = result_queue
        self.__secret_key = config['key']
        self.__mode = config['mode']

    def run(self):
        # print(f"Worker {self.worker_id} started with mode {self.__mode}")
        while True:
            try:
                task = self.task_queue.get()
                if task == 'TERMINATE':
                    # print(f"Worker {self.worker_id} terminating")
                    break
                result = self.__process(task)
                self.result_queue.put({'worker': self.worker_id, 'result': result, 'task': task})
            except Exception as e:
                self.result_queue.put({'worker': self.worker_id, 'error': str(e), 'task': task})

    def __process(self, task):
        if self.__mode == 'encrypt':
            return f"{task}_{self.__secret_key}"
        elif self.__mode == 'hash':
            return hash(task + self.__secret_key)
        else:
            raise ValueError("Invalid mode")

def worker_process(task_queue, result_queue):
    worker_id = multiprocessing.current_process()._identity[0]
    config = {
        'key': f"KEY{worker_id}",
        'mode': 'encrypt' if worker_id % 2 == 0 else 'hash'
    }
    worker = CustomWorker(worker_id, task_queue, result_queue, config)
    worker.run()  

def main():
    manager = Manager()
    task_queue = manager.Queue()
    result_queue = manager.Queue()

    pool_size = 3
    
    # 先放入所有任务
    for i in range(20):
        task_queue.put(f"task_{i}")
    
    # 添加足够的终止信号
    for _ in range(pool_size):
        task_queue.put("TERMINATE")

    # 创建并启动进程池
    pool = Pool(
        processes=pool_size,
        initializer=worker_process,
        initargs=(task_queue, result_queue)
    )
    
    # 收集结果
    results = []
    tasks_processed = 0
    expected_tasks = 20  # 期望处理的任务数
    
    while tasks_processed < expected_tasks:
        result = result_queue.get()
        tasks_processed += 1
        
        if 'error' in result:
            print(f"Error from worker {result['worker']} processing {result['task']}: {result['error']}")
        else:
            results.append(result)
            # print(f"Worker {result['worker']} processed {result['task']} → {result['result']}")
    
    # 等待所有进程完成
    pool.close()
    pool.join()
    
    # print(f"Processed {len(results)} tasks successfully")
    print(f"Final results: {results}")

if __name__ == '__main__':
    main()

Overwriting multiprocessing_script.py


In [4]:
import subprocess  

for i in range(4):
    print(f"Running subprocess {i}")
    result = subprocess.run(
        ["python", "multiprocessing_script.py"], 
        capture_output=True, text=True)
    print("STDOUT:", result.stdout)
    print("STDERR:", result.stderr)  # 查看是否有错误

Running subprocess 0
STDOUT: Final results: [{'worker': 2, 'result': 'task_0_KEY2', 'task': 'task_0'}, {'worker': 2, 'result': 'task_1_KEY2', 'task': 'task_1'}, {'worker': 2, 'result': 'task_2_KEY2', 'task': 'task_2'}, {'worker': 2, 'result': 'task_3_KEY2', 'task': 'task_3'}, {'worker': 2, 'result': 'task_4_KEY2', 'task': 'task_4'}, {'worker': 2, 'result': 'task_5_KEY2', 'task': 'task_5'}, {'worker': 2, 'result': 'task_6_KEY2', 'task': 'task_6'}, {'worker': 2, 'result': 'task_7_KEY2', 'task': 'task_7'}, {'worker': 2, 'result': 'task_8_KEY2', 'task': 'task_8'}, {'worker': 2, 'result': 'task_9_KEY2', 'task': 'task_9'}, {'worker': 2, 'result': 'task_10_KEY2', 'task': 'task_10'}, {'worker': 2, 'result': 'task_11_KEY2', 'task': 'task_11'}, {'worker': 2, 'result': 'task_12_KEY2', 'task': 'task_12'}, {'worker': 2, 'result': 'task_13_KEY2', 'task': 'task_13'}, {'worker': 2, 'result': 'task_14_KEY2', 'task': 'task_14'}, {'worker': 2, 'result': 'task_15_KEY2', 'task': 'task_15'}, {'worker': 2, '

此程序整体运行流程：
1.主进程生成任务，放入共享队列
2.3 个子进程并发从队列取任务，自动区分处理模式
3.处理完成后，结果通过队列回传给主进程
4.主进程收满 20 个结果后，子进程收到终止信号退出
5.主进程打印所有任务结果

## **0.2** 多线程与线程锁

需要判断
- 输出是否稳定（即多次运行是否能保证输出结果一致）
  - 如果稳定，预测输出结果
  - 如果不稳定，预测所有输出结果的可能情况
- 两段代码平均运行时间的关系

In [5]:
import threading
import time
import random

In [6]:
result = []

def append_numbers1(thread_id):
    for i in range(3):
        time.sleep(random.random())
        result.append(f'{i} from thread {thread_id}')
        

threads = [threading.Thread(
    target=append_numbers1, args=(i, )) for i in range(3)]
for t in threads:
    t.start()
for t in threads:
    t.join()

print(result)

['0 from thread 2', '0 from thread 0', '0 from thread 1', '1 from thread 2', '2 from thread 2', '1 from thread 0', '2 from thread 0', '1 from thread 1', '2 from thread 1']


Python 列表的 append() 方法是线程安全的原子操作，不会出现数据丢失 / 错乱；
因为 random.sleep() 随机休眠，线程执行顺序不固定，最终 result 里的元素顺序随机；
3 个线程各追加 3 个数据，最终列表一定包含 9 个元素

## **0.3** await方法实现协程



In [7]:
import sys
from io import StringIO
import asyncio

class TreeNode:
    '''二叉搜索树节点的定义'''
    def __init__(self, val):
        self.val = val
        self.left = None
        self.right = None

class OperationTree:
    '''二叉搜索树操作'''
    def insert(self, root, val):
        '''二叉搜索树插入操作'''
        if root == None:
            root = TreeNode(val)
        elif val < root.val:
            root.left = self.insert(root.left, val)
        elif val > root.val:
            root.right = self.insert(root.right, val)
        return root

async def inorder_collect(root, result_list):
    '''异步递归收集结果'''
    if root:
        await inorder_collect(root.left, result_list)
        await asyncio.sleep(0)  # 出让控制避免单任务独占
        result_list.append(root.val)
        await inorder_collect(root.right, result_list)

async def main_collect():
    op = OperationTree()
    root = None
    values = [5, 3, 7, 2, 4, 6, 8]
    for val in values:
        root = op.insert(root, val)
    
    result = []
    await inorder_collect(root, result)
    print("异步收集结果：")
    for val in result:
        print(val, end=" ")
    print()
    
await(main_collect())

异步收集结果：
2 3 4 5 6 7 8 


异步IO使得程序的运行效率大幅提升，但也带来了新的复杂性。

### 0.4 网络服务 



In [ ]:
import asyncio
import threading
import time

# ========== 超简短HTTP服务器 ==========
async def quick_http_server():
    """HTTP服务器，运行3秒后自动关闭"""
    
    async def handler(reader, writer):
        # 读取请求
        data = await reader.read(1024)
        request = data.decode()
        
        # 解析请求路径
        path = request.split(' ')[1] if ' ' in request else '/'
        print(f"请求: {path}")
        
        # 根据路径返回不同内容
        if path == '/':
            body = "欢迎访问根路径！"
        elif path == '/hello':
            body = "Hello, World!"
        else:
            body = "404 Not Found"
        
        # 构造HTTP响应
        response = (
            f"HTTP/1.1 200 OK\r\n"
            f"Content-Length: {len(body)}\r\n"
            f"\r\n"
            f"{body}"
        )
        
        writer.write(response.encode())
        await writer.drain()
        writer.close()
    
    # 启动服务器
    server = await asyncio.start_server(handler, 'localhost', 8080)
    print("✅ HTTP服务器启动 (3秒后关闭)")
    print("访问: http://localhost:8080/hello")
    
    await asyncio.sleep(3)
    server.close()
    print("❌ 服务器关闭")

# ========== 超简短HTTP客户端 ==========
async def http_get(path):
    """发送GET请求"""
    try:
        reader, writer = await asyncio.open_connection('localhost', 8080)
        
        # 发送HTTP请求
        request = f"GET {path} HTTP/1.1\r\nHost: localhost\r\n\r\n"
        writer.write(request.encode())
        await writer.drain()
        
        # 接收响应
        response = await reader.read(1024)
        # 提取响应体（去掉HTTP头）
        body = response.decode().split('\r\n\r\n', 1)[1]
        print(f"响应: {body}")
        
        writer.close()
        await writer.wait_closed()
        
    except Exception as e:
        print(f"请求失败: {e}")

# ========== 运行 ==========
print("="*40)

# 启动服务器
threading.Thread(target=lambda: asyncio.run(quick_http_server()), daemon=True).start()
await asyncio.sleep(1)

# 发送请求
print("\n发送请求:")
await http_get('/')
await http_get('/hello')
await http_get('/notexist')

# 等待服务器关闭
await asyncio.sleep(2)
print("\n尝试访问关闭的服务器:")
await http_get('/')

print("\n完成!")

# 第一部分 基础练习 补充代码完善功能

## **1.1** 进程的创建

In [8]:
%%writefile multiprocessing_script_task1_1.py
#导入模块
import multiprocessing
import time
 
#创建进程调用函数
def work1(interval):
	print('执行work1')
	time.sleep(interval)
	print('end work1')
 
def work2(interval):
	print('执行work2')
	time.sleep(interval)
	print('end work2')
 
if __name__ == "__main__":
	print('执行主进程')
	#代码填空：创建进程对象
	p1=multiprocessing.Process(target=work1,args=(2,))
	p2=multiprocessing.Process(target=work2,args=(3,))
	#代码填空：启动进程
	p1.start()
	p2.start()
	p1.join()
	p2.join()
	print('主进程结束')

Writing multiprocessing_script_task1_1.py


In [9]:
import subprocess
result = subprocess.run(
    ["python", "multiprocessing_script_task1_1.py"], 
    capture_output=True, text=True)
assert "执行主进程" in result.stdout
assert "执行work1" in result.stdout
assert "执行work2" in result.stdout
assert "end work1" in result.stdout
assert "end work2" in result.stdout
assert "主进程结束" in result.stdout

## **1.2** 进程池

In [14]:
%%writefile multiprocessing_script_task1_2.py
import multiprocessing

def square(x):
    return x * x

if __name__ == '__main__':
    # 代码填空：创建一个进程池，进程数为4
    with multiprocessing.Pool(4) as pool:
        results = [pool.apply_async(square, (i,)) for i in range(5)]
        output = [res.get() for res in results]
    print(output)  # 应该输出 [0, 1, 4, 9, 16]

Overwriting multiprocessing_script_task1_2.py


In [15]:
import subprocess
result = subprocess.run(
    ["python", "multiprocessing_script_task1_2.py"], 
    capture_output=True, text=True)
assert result.stdout == "[0, 1, 4, 9, 16]\n"

## **1.3** 进程消息传递

In [16]:
%%writefile multiprocessing_script_task1_3.py
import multiprocessing

def consumer(q):
    while True:
        # 代码填空：从队列中获取数据
        item =q.get()
        if item == 'STOP':
            break
        print(f"消费: {item}")

def producer(q):
    for i in range(3):
        # 代码填空：向队列中添加数据，内容为"产品i"
        q.put(f"产品{i}")

    q.put('STOP')

if __name__ == '__main__':
    # 代码填空：创建一个队列
    q = multiprocessing.Queue()
    p1 = multiprocessing.Process(target=producer, args=(q,))
    p2 = multiprocessing.Process(target=consumer, args=(q,))
    p1.start()
    p2.start()
    p1.join()
    p2.join()

Writing multiprocessing_script_task1_3.py


In [17]:
import subprocess
result = subprocess.run(
    ["python", "multiprocessing_script_task1_3.py"], 
    capture_output=True, text=True)
# print(result.stdout)
assert "消费: 产品0" in result.stdout
assert "消费: 产品1" in result.stdout
assert "消费: 产品2" in result.stdout


## **1.4** 线程的创建和传参

创建两个线程，分别计算`x`, `y`的和与积，存入`result`中

In [18]:
import threading
x, y = 3, 4

result = {}
def worker1(x, y):
    result['add'] = x + y
    
def worker2(x, y):
    result['multiply'] = x * y

t1=threading.Thread(target=worker1, args=(x, y))
t2=threading.Thread(target=worker2, args=(x, y))
# t1 = ... # TODO: create a thread
# t2 = ... # TODO: create a thread

t1.start()
t2.start()
t1.join()
t2.join()

assert result == {'add': 7, 'multiply': 12}

## **1.5** 线程锁

在以下代码用`#######`包含的部分中，在合适的位置添加和应用线程锁，保证`counter`线程安全

In [20]:
import threading
import time
import random

counter = []
lock = threading.Lock()  # 创建锁对象

##########################################
# TODO: add and apply a lock
# 给counter列表加锁，确保线程安全

def worker():
    for _ in range(100):
        with lock:  # 使用锁对象
            counter.append(1)
            counter[-1] += 1
        time.sleep(random.random() * 0.01)

##########################################

threads = [threading.Thread(target=worker) for _ in range(10)]
for t in threads:
    t.start()
for t in threads:
    t.join()

assert counter == [2] * 1000

## **1.6** 协程编程

In [ ]:
import nest_asyncio
nest_asyncio.apply() 

import asyncio
import random

async def factory(transfer_center, products):
    '''
    参数说明：
        - transfer_center: 是一个Queue, 用于模拟转运中心
        - 其中：put方法用于产品放入队列，get方法取出消费一个产品
        - products: 是一个元素为str的列表, 表示工厂所需生产的所有产品名称
    '''

    ###############################
    #--- Your code starts here ---#
    ###############################
    for product in products:
        # 1. 生成1~3秒的随机数表示生产用时，使用asyncio.sleep模拟生产过程
        await asyncio.sleep(random.randint(1, 3))
        # 2. 打印下面这句话："工厂生产：<产品名字>"
        print('工厂生产：' + product)
        # 3. 将产品加入到转运中心
        await transfer_center.put(product)
        # 4. 通知队列该任务已完成
    transfer_center.task_done()

    # TODO: 补充作为生产者的factory的行为
    # 对每一个产品，
    #       1. 生成1~3秒的随机数表示生产用时，使用asyncio.sleep模拟生产过程
    #       2. 打印下面这句话："工厂生产：<产品名字>"
    #       3. 将产品加入到转运中心    
    # 生产完所有产品后，在队列里放置一个结束标志
    # None 作为结束标志

    ###############################
    #---  Your code ends here  ---#
    ###############################
    
    print('工厂生产完毕')
    await transfer_center.join()
    

async def supermarket(transfer_center):

    while True:

        ###############################
        #--- Your code starts here ---#
        ###############################
        product = await transfer_center.get()
        if product is None:
             break
        # 1. 生成1~3秒的随机数表示接收用时，使用asyncio.sleep模拟接受过程
        await asyncio.sleep(random.randint(1, 3))
        # 2. 打印下面这句话："超市接收：<产品名字>"
        print('超市接收：' + product)
        # 3. 从转运中心获取产品
        # 4. 添加退出循环的判断

        # TODO: 补充作为消费者的supermarket的行为
        # 对每一个产品，
        #       1. 生成1~3秒的随机数表示接收用时，使用asyncio.sleep模拟接受过程
        #       2. 打印下面这句话："超市接收：<产品名字>"
        #       3. 从转运中心获取产品
        #      4. 添加退出循环的判断
        
        
        ###############################
        #---  Your code ends here  ---#
        ###############################
    # 3. 通知队列该任务已完成
        transfer_center.task_done()

    print('超市接收完毕')


transfer_center = asyncio.Queue()

factory_crt = factory(transfer_center, ['商品' + str(i) for i in range(1, 11)])
supermarket_crt = supermarket(transfer_center)

loop = asyncio.get_event_loop()
task1 = loop.create_task(factory_crt)
task2 = loop.create_task(supermarket_crt) #原代码有问题，这里协程对象是不能await的，需要传入tasks列表
tasks = [task1,task2]
_ = loop.run_until_complete(asyncio.wait(tasks))


# 以上实现的是工厂生产用时与超市接受用时均值相同时的情况
# 观察输出，可以发现工厂和超市的输出是近似交替出现的
# 可以思考一下当生产用时大于/小于接受用时的时候，输出情况将会如何
# 可以修改上面代码中随机生成时间的范围并运行，验证你的猜想

工厂生产：商品1
超市接收：商品1
工厂生产：商品2
超市接收：商品2
工厂生产：商品3
超市接收：商品3
工厂生产：商品4
工厂生产：商品5
超市接收：商品4
工厂生产：商品6
超市接收：商品5
超市接收：商品6
工厂生产：商品7
工厂生产：商品8
超市接收：商品7
超市接收：商品8
工厂生产：商品9
超市接收：商品9
工厂生产：商品10
工厂生产完毕
超市接收：商品10


### 第二部分 代码-算法阅读理解

下面的学生选课的代码实现（by： 王羲羽 2300015408）

以及后面由deepseek做出的三分代码实现的对比评价（陈甄 2500010701）（贾富淼 2400015457）。

选做：用B/S方案实现学生选课系统，在服务器端用线程池实现并发，在客户端用协程实现对选课结果的动态返回和重新补-退选。可以由AI辅助完成。完成并自己读懂AI代码，写出简短的模型框架和算法思路。完成的同学可以给自己的作业加一个#

#### 发布：订阅模式
上星期的课程注册系统添加功能，成为了一个发布：订阅模式。学生作为订阅者，注册自己的学号、姓名以及意向选课列表。类似初选过程。等所有学生完成选课后，进行优化，优化器如下所述。等到优化完成后，选上课的学生的enrollment status会变成success，并增加课程列表。同时课程系统的rosters会添加学生子类。后续当课程发布消息的时候，在roster上的学生会收到消息。一些实现案例在后面。


#### 优化器：

优化器依然同上次作业，采用被动算法。我的系统采用了经济学边际效用递减的原理，更好平衡了公平性。由于课程和人数均较少，因此采取01规划获得最优解，当数据量增大可以采用贪心。在学生进行初次选课（可以超过6学分）后，我建立以下的线形规划。

1. 决策变量
$$x_{i,j} = 
\begin{cases} 
1, & \text{若学生 } i \text{ 选修课程 } j \\
0, & \text{反之}
\end{cases}
\quad \forall i \in \{1, \dots, n\}, j \in \{1, \dots, m\}$$

2. 目标函数

$$\text{Maximize } Z = \sum_{i=1}^{n} \ln \left( \sum_{j=1}^{m} x_{i,j} \cdot c_j + 1 \right)$$

即用一个ln(x+1)的函数来衡量一个最终上x分课程的学生的收益

3. 约束条件 (Constraints)$$\begin{aligned}
\text{s.t.} \quad & \sum_{j=1}^{m} x_{i,j} \cdot c_j \le 6, & \forall i \in \{1, \dots, n\} \quad & \text{(个体学分上限)} \\
& \sum_{i=1}^{n} x_{i,j} \le 3, & \forall j \in \{1, \dots, m\} \quad & \text{(课程容量上限)} \\
& x_{i,j} \in \{0, 1\}, & \forall i, j \quad & \text{(变量二元约束)} \\
& x_{i,j} = 0, & \text{if } j \notin I_i \quad & \text{(选课意向约束)}
\end{aligned}$$

这个思路很直接，对数函数 $\ln(x+1)$ 的导数（边际效用）随 $x$ 增大而减小。这意味着将学分从 0 提升到 2 的贡献约为 1.10。将学分从 4 提升到 6 的社会福利贡献约为 0.33，系统会自发地“劫富济贫”，在面临同一门课的竞争的时候会优先保障没有学分的人选上课。

在算法上，01暴力枚举的时间复杂度最高为2^(N*M),这里做一些优化：首先遍历每一个学生选择的课程的组合，筛选出总学分<=6的组合组成选择包裹。随后对于每一个人的包裹进行深度优先搜索，在发现有课程选课人数大于3后剪枝回溯，极大减少计算量。

In [ ]:
import math
from enum import Enum
from abc import ABC, abstractmethod
from typing import List, Tuple, Dict, Set
from itertools import combinations

# ==========================================
# 1. 状态定义
# ==========================================
class EnrollmentStatus(Enum):
    PENDING = "待定 (Pending)"
    SUCCESS = "选课成功 (Success)"
    FAILED = "选课失败 (Failed)"

# ==========================================
# 2. 观察者接口 (Observer Interface)
# ==========================================
class IStudent(ABC):
    @property
    @abstractmethod
    def student_id(self) -> str: pass
    
    @abstractmethod
    def update_result(self, status: EnrollmentStatus, courses: List[str], credits: int): pass

    @abstractmethod
    def receive_assignment(self, course_name: str, task: str): pass

# ==========================================
# 3. 具体学生类 (Concrete Observer)
# ==========================================
class Student(IStudent):
    def __init__(self, student_id: str, name: str):
        self._id = student_id
        self.name = name
        self.status = EnrollmentStatus.PENDING
        self.final_courses: List[str] = []
        self.total_credits = 0

    @property
    def student_id(self) -> str: return self._id

    def update_result(self, status: EnrollmentStatus, courses: List[str], credits: int):
        """系统结算后，同步更新观察者的内部状态"""
        self.status = status
        self.final_courses = courses
        self.total_credits = credits

    def receive_assignment(self, course_name: str, task: str):
        """仅限选课成功的学生触发此回调"""
        print(f"  [📩 {self.name} 收到信息]: 《{course_name}》-> {task}")

    def __lt__(self, other):
        return self._id < other._id

    def __repr__(self):
        return f"Student('{self._id}', '{self.name}', status={self.status.value})"

# ==========================================
# 4. 选课系统 (Subject & Optimizer)
# ==========================================
class CourseRegistrationSystem:
    def __init__(self, courses_data: List[Tuple[str, int]], capacity: int = 3, student_limit: int = 6):
        self.credits_map = dict(courses_data)
        self.capacity_limit = capacity
        self.student_limit = student_limit
        
        # 维护观察者名单与意向
        self._students: Dict[str, Student] = {}
        self._intentions: Dict[str, List[str]] = {}
        
        # 结果存储与精准推送名单
        self._rosters: Dict[str, List[Student]] = {c: [] for c, _ in courses_data}
        self.best_utility = -1.0
        self.best_assignment: Dict[str, Tuple[List[str], int]] = {}

    def attach(self, student: Student, wishes: List[str]):
        """订阅：学生加入系统并提交原始意向"""
        self._students[student.student_id] = student
        self._intentions[student.student_id] = wishes

    def _get_bundles(self, wishes: List[str]):
        """局部预处理：计算每个学生的合法学分组合"""
        bundles = [([], 0.0, 0)]
        for r in range(1, len(wishes) + 1):
            for combo in combinations(wishes, r):
                total_c = sum(self.credits_map[c] for c in combo)
                if total_c <= self.student_limit:
                    # 效用函数 ln(x+1)
                    bundles.append((list(combo), math.log(total_c + 1), total_c))
        return sorted(bundles, key=lambda x: x[1], reverse=True)

    def solve_and_notify(self):
        """全局优化：执行 0-1 规划搜索并通知观察者"""
        print("--- 系统：正在计算全局最优解 (0-1 Non-linear Programming) ---")
        
        s_ids = sorted(self._students.keys())
        choices_pool = {sid: self._get_bundles(self._intentions[sid]) for sid in s_ids}
        
        counts = {c: 0 for c in self.credits_map}
        temp_assign = {}

        def backtrack(idx, current_util):
            if idx == len(s_ids):
                if current_util > self.best_utility:
                    self.best_utility = current_util
                    self.best_assignment = temp_assign.copy()
                return

            sid = s_ids[idx]
            for combo, util, cred in choices_pool[sid]:
                # 检查课程容量限制
                if all(counts[c] < self.capacity_limit for c in combo):
                    # 做出选择
                    for c in combo: counts[c] += 1
                    temp_assign[sid] = (combo, cred)
                    
                    # 递归
                    backtrack(idx + 1, current_util + util)
                    
                    # 回溯
                    for c in combo: counts[c] -= 1
                    del temp_assign[sid]

        # 启动搜索
        backtrack(0, 0.0)

        # 结算阶段：更新观察者状态并建立精准订阅名单
        for sid, student in self._students.items():
            res_courses, res_credits = self.best_assignment.get(sid, ([], 0))
            if res_courses:
                student.update_result(EnrollmentStatus.SUCCESS, res_courses, res_credits)
                # 将学生拉入对应的“课群”观察者名单
                for c in res_courses:
                    self._rosters[c].append(student)
            else:
                student.update_result(EnrollmentStatus.FAILED, [], 0)
        
        print(f"结算完成！最大社会福利: {self.best_utility:.4f}\n")

    def push_assignment(self, course_name: str, task: str):
        """精准推送：仅限在 _rosters[course_name] 中的观察者收到"""
        print(f"[📢 系统发布] 《{course_name}》信息内容：{task}")
        subscribers = self._rosters.get(course_name, [])
        if not subscribers:
            print(f"  (无人选修 {course_name}，无需推送)")
            return
        for student in subscribers:
            student.receive_assignment(course_name, task)

# ==========================================
# 5. 运行测试 (复刻高压竞争场景)
# ==========================================
def run_test():
    # 1. 课程定义
    COURSES_DATA = [('三宝', 2), ('音数', 2), ('地概', 2), ('高数', 4), ('线代', 3), ('棒垒', 1)]
    
    # 2. 初始化选课系统 (每门课容量限制 3 人，学生最高限 6 学分)
    system = CourseRegistrationSystem(COURSES_DATA, capacity=3, student_limit=6)
    
    # 3. 创建学生观察者
    student_instances = [
        Student("202401", "张三"), Student("202402", "李四"),
        Student("202403", "王五"), Student("202404", "赵六"),
        Student("202405", "钱七"), Student("202406", "孙八")
    ]
    
    # 4. 设置意向 (9/8/7/8/5/8 分意向，制造激烈竞争)
    intentions = {
        "202401": ["高数", "线代", "三宝"], # 9分
        "202402": ["高数", "音数", "地概"], # 8分
        "202403": ["线代", "地概", "三宝"], # 7分
        "202404": ["高数", "线代", "棒垒"], # 8分
        "202405": ["音数", "地概", "棒垒"], # 5分
        "202406": ["高数", "三宝", "音数"]  # 8分
    }

    # 5. 提交申请 (Attach)
    for s in student_instances:
        system.attach(s, intentions[s.student_id])

    # 6. 全局优化并状态变更 (Notify)
    system.solve_and_notify()

    # 7. 打印结果表
    print("="*75)
    print(f"{'学号':<8} | {'姓名':<4} | {'状态':<15} | {'学分':<4} | {'最终选课名单'}")
    print("-" * 75)
    for s in sorted(student_instances):
        print(f"{s.student_id:<8} | {s.name:<4} | {s.status.value:<15} | {s.total_credits:<4} | {s.final_courses}")
    
    print("-" * 75)
    print(f"全局最大社会福利值 (Σ ln(x+1)): {system.best_utility:.4f}")
    print("="*75)

    # 8. 精准作业分发测试
    print("\n--- 动态通知测试 ---")
    system.push_assignment("高数", "求解 Black-Scholes 方程的数值解")
    system.push_assignment("三宝", "提交故宫博物院观后感")
    system.push_assignment("棒垒", "明天下午 2:00 操场集合")

if __name__ == "__main__":
    run_test()

该系统采用 **B/S 架构**，通过 **生产者-消费者模型** 实现高并发选课：

### 1. 服务端模型
- **并发处理**：`ThreadPoolExecutor` 线程池隔离业务逻辑，避免阻塞 Flask 主线程；  
- **数据安全**：`threading.Lock` 保护课程容量、学生学分等共享资源，确保原子性操作，防止超卖；  
- **算法流程**：
  - **选课**：校验（存在性、是否已选、余量、学分）→ 加锁 → 更新课程人数、学生学分、选课列表 → 释放锁；  
  - **退选**：校验 → 加锁 → 人数回减、学分返还、移除记录 → 释放锁。  
- **动态查询**：`/status` 和 `/my_info` 直接返回快照，无锁读取。

### 2. 客户端模型
- **异步 I/O**：基于 `asyncio` + `httpx` 的协程客户端，非阻塞发起请求，提升并发请求效率；  
- **实时反馈**：通过 `await` 挂起等待响应，根据返回的 `msg` 字段提示成功或失败原因（学分不足、名额已满等），实现友好交互。

### 3. 核心设计思想
- **资源隔离**：线程池处理耗时操作，前端协程高效发送请求；  
- **一致性保障**：互斥锁确保多线程环境下数据的正确性；  
- **可扩展性**：后端可通过修改线程池大小、增加节点横向扩展；前端可轻松增加并发请求量。

此方案模拟了真实选课系统中应对瞬时高并发的典型策略。

In [27]:
%%writefile course_system.py
# 将上面的 course_system.py 完整代码粘贴到此处（注意缩进）
"""
课程选课系统 - 整合版
支持通过命令行参数选择运行服务端或客户端
"""

import sys
import time
import threading
import asyncio
from concurrent.futures import ThreadPoolExecutor

# 服务端依赖
try:
    from flask import Flask, request, jsonify
except ImportError:
    print("服务端需要 Flask 库，请执行：pip install flask")
    sys.exit(1)

# 客户端依赖
try:
    import httpx
except ImportError:
    print("客户端需要 httpx 库，请执行：pip install httpx")
    sys.exit(1)

# ==================== 服务端代码 ====================
app = Flask(__name__)

# 模拟数据库
courses = {
    "C01": {"name": "Python并发编程", "capacity": 3, "current": 0, "credits": 2},
    "C02": {"name": "微服务架构", "capacity": 2, "current": 0, "credits": 3},
    "C03": {"name": "大学物理", "capacity": 10, "current": 0, "credits": 4}
}

students = {
    "S001": {"name": "小明", "selected": [], "credits_left": 5}
}

data_lock = threading.Lock()
executor = ThreadPoolExecutor(max_workers=5)


@app.route('/status', methods=['GET'])
def get_status():
    """查询所有课程现有人数"""
    return jsonify({"courses": courses})


@app.route('/my_info', methods=['GET'])
def get_my_info():
    """查询个人已选课程和剩余学分"""
    sid = request.args.get("sid")
    student = students.get(sid)
    if not student:
        return jsonify({"error": "学生不存在"}), 404
    return jsonify(student)


def process_add(sid, cid):
    with data_lock:
        student = students.get(sid)
        course = courses.get(cid)

        if not student or not course:
            return {"status": "失败", "msg": "学生或课程ID无效"}
        if cid in student['selected']:
            return {"status": "失败", "msg": "已选过该课程"}
        if course['current'] >= course['capacity']:
            return {"status": "失败", "msg": "课程名额已满"}
        if student['credits_left'] < course['credits']:
            return {"status": "失败", "msg": "学分不足，无法补选"}

        # 模拟处理延迟
        time.sleep(0.5)

        course['current'] += 1
        student['selected'].append(cid)
        student['credits_left'] -= course['credits']
        return {"status": "成功", "msg": f"补选 {course['name']} 成功"}


def process_drop(sid, cid):
    with data_lock:
        student = students.get(sid)
        course = courses.get(cid)

        if not student or cid not in student['selected']:
            return {"status": "失败", "msg": "未选修此课程，无法退选"}

        time.sleep(0.3)

        course['current'] -= 1
        student['selected'].remove(cid)
        student['credits_left'] += course['credits']
        return {"status": "成功", "msg": f"退选 {course['name']} 成功"}


@app.route('/add', methods=['POST'])
def add_course():
    data = request.json
    future = executor.submit(process_add, data['sid'], data['cid'])
    return jsonify(future.result())


@app.route('/drop', methods=['POST'])
def drop_course():
    data = request.json
    future = executor.submit(process_drop, data['sid'], data['cid'])
    return jsonify(future.result())


def run_server():
    """启动服务端"""
    print("服务器启动：处理选课线程池已就绪...")
    app.run(port=5000)


# ==================== 客户端代码 ====================
SERVER_URL = "http://127.0.0.1:5000"
STUDENT_ID = "S001"


async def fetch_status(client):
    """协程：查询课程现状"""
    resp = await client.get(f"{SERVER_URL}/status")
    data = resp.json()
    print("\n--- 课程当前状态 ---")
    for cid, info in data['courses'].items():
        print(f"ID:{cid} | {info['name']} | 人数:{info['current']}/{info['capacity']} | 学分:{info['credits']}")


async def fetch_my_info(client):
    """协程：查询个人信息"""
    resp = await client.get(f"{SERVER_URL}/my_info", params={"sid": STUDENT_ID})
    data = resp.json()
    print(f"\n[我的信息] 已选代码: {data['selected']} | 剩余学分: {data['credits_left']}")


async def operate_course(client, action, cid):
    """协程：执行补/退选动作"""
    url = f"{SERVER_URL}/{action}"
    payload = {"sid": STUDENT_ID, "cid": cid}
    resp = await client.post(url, json=payload)
    result = resp.json()
    print(f"\n>>> 操作结果 [{action}]: {result['status']} - {result['msg']}")


async def main_client():
    """客户端主协程"""
    async with httpx.AsyncClient() as client:
        while True:
            print("\n========== 学生选课系统 (协程客户端) ==========")
            print("1. 查看课程表  2. 补选课程  3. 退选课程  4. 查看我的课表  5. 退出")
            choice = input("请输入操作编号: ")

            if choice == '1':
                await fetch_status(client)
            elif choice == '2':
                cid = input("输入要补选的课程ID: ")
                await operate_course(client, "add", cid)
            elif choice == '3':
                cid = input("输入要退选的课程ID: ")
                await operate_course(client, "drop", cid)
            elif choice == '4':
                await fetch_my_info(client)
            elif choice == '5':
                break
            else:
                print("无效输入")


def run_client():
    """启动客户端"""
    try:
        asyncio.run(main_client())
    except KeyboardInterrupt:
        print("\n客户端退出")


# ==================== 主入口 ====================
if __name__ == "__main__":
    if len(sys.argv) < 2:
        print("请指定运行模式：server 或 client")
        print("示例：")
        print("  python course_system.py server   # 启动服务端")
        print("  python course_system.py client   # 启动客户端")
        sys.exit(1)

    mode = sys.argv[1].lower()
    if mode == "server":
        run_server()
    elif mode == "client":
        run_client()
    else:
        print("无效模式，请使用 server 或 client")
        run_server()
        sys.exit(1)

Writing course_system.py


In [28]:
import threading
import time
import requests

# 导入服务端主函数（注意：直接导入会执行 Flask 应用，需要先处理）
# 因为 course_system.py 中使用了 if __name__ == "__main__"，所以不能直接导入 run_server
# 但我们可以通过子进程启动，或者修改代码后导入。这里推荐用子进程。

import subprocess
import sys

# 启动服务端子进程
server_process = subprocess.Popen(
    [sys.executable, "course_system.py", "server"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

# 等待服务器启动
time.sleep(2)
print("服务端已启动，PID:", server_process.pid)

服务端已启动，PID: 3596


In [29]:
import requests

BASE_URL = "http://127.0.0.1:5000"
SID = "S001"

def print_response(response):
    print(f"状态码: {response.status_code}")
    print(f"响应内容: {response.json()}")

# 1. 查看课程状态
print("\n=== 查看所有课程 ===")
resp = requests.get(f"{BASE_URL}/status")
print_response(resp)

# 2. 查看我的信息（初始）
print("\n=== 查看我的信息（初始） ===")
resp = requests.get(f"{BASE_URL}/my_info", params={"sid": SID})
print_response(resp)

# 3. 补选课程 C01（应成功）
print("\n=== 补选 C01 ===")
resp = requests.post(f"{BASE_URL}/add", json={"sid": SID, "cid": "C01"})
print_response(resp)

# 4. 补选课程 C02（应成功）
print("\n=== 补选 C02 ===")
resp = requests.post(f"{BASE_URL}/add", json={"sid": SID, "cid": "C02"})
print_response(resp)

# 5. 补选课程 C03（学分不足，应失败）
print("\n=== 补选 C03（学分不足） ===")
resp = requests.post(f"{BASE_URL}/add", json={"sid": SID, "cid": "C03"})
print_response(resp)

# 6. 再次查看课程状态
print("\n=== 查看所有课程（更新后） ===")
resp = requests.get(f"{BASE_URL}/status")
print_response(resp)

# 7. 查看我的信息（更新后）
print("\n=== 查看我的信息（更新后） ===")
resp = requests.get(f"{BASE_URL}/my_info", params={"sid": SID})
print_response(resp)

# 8. 退选课程 C02
print("\n=== 退选 C02 ===")
resp = requests.post(f"{BASE_URL}/drop", json={"sid": SID, "cid": "C02"})
print_response(resp)

# 9. 再次查看课程状态和我的信息
print("\n=== 查看所有课程（退选后） ===")
resp = requests.get(f"{BASE_URL}/status")
print_response(resp)

print("\n=== 查看我的信息（退选后） ===")
resp = requests.get(f"{BASE_URL}/my_info", params={"sid": SID})
print_response(resp)


=== 查看所有课程 ===
状态码: 200
响应内容: {'courses': {'C01': {'capacity': 3, 'credits': 2, 'current': 0, 'name': 'Python并发编程'}, 'C02': {'capacity': 2, 'credits': 3, 'current': 0, 'name': '微服务架构'}, 'C03': {'capacity': 10, 'credits': 4, 'current': 0, 'name': '大学物理'}}}

=== 查看我的信息（初始） ===
状态码: 200
响应内容: {'credits_left': 5, 'name': '小明', 'selected': []}

=== 补选 C01 ===
状态码: 200
响应内容: {'msg': '补选 Python并发编程 成功', 'status': '成功'}

=== 补选 C02 ===
状态码: 200
响应内容: {'msg': '补选 微服务架构 成功', 'status': '成功'}

=== 补选 C03（学分不足） ===
状态码: 200
响应内容: {'msg': '学分不足，无法补选', 'status': '失败'}

=== 查看所有课程（更新后） ===
状态码: 200
响应内容: {'courses': {'C01': {'capacity': 3, 'credits': 2, 'current': 1, 'name': 'Python并发编程'}, 'C02': {'capacity': 2, 'credits': 3, 'current': 1, 'name': '微服务架构'}, 'C03': {'capacity': 10, 'credits': 4, 'current': 0, 'name': '大学物理'}}}

=== 查看我的信息（更新后） ===
状态码: 200
响应内容: {'credits_left': 0, 'name': '小明', 'selected': ['C01', 'C02']}

=== 退选 C02 ===
状态码: 200
响应内容: {'msg': '退选 微服务架构 成功', 'status': '成功'}

=== 查看所有

In [30]:
server_process.terminate()
server_process.wait()
print("服务端已停止")

服务端已停止


In [ ]:
选了三份比较典型的代码实现让Deepseek做的对比评价：

六、综合评价
第一份代码（抽签+背包）：8/10
优点：实用、可扩展

缺点：观察者模式没有实现信息推送，背包算法在抽签后会导致资源浪费

第二份代码（全局0-1规划）：9/10
优点：架构优雅，精准推送，学分越多，额外学分的边际效用越低避免学生过度选课

缺点：指数级算法复杂度，完全确定性可能带来的永远选不上这种极端情况

第三份代码（Gumbel-max+ILP+观察者）：9.8/10
优势总结：

✅ 算法最优：Gumbel-max保证公平性，ILP保证最优性

✅ 可扩展性强：剪枝+可选pulp求解器，支持大规模

✅ 观察者模式完整：课程独立管理订阅，支持退订

✅ 理论扎实：拍卖理论+随机分配+极值分布

✅ 工程完善：稳定性测试、统计输出、错误处理

✅ 经济学正确：投点决定概率，不是完全随机

唯一不足：

代码复杂度较高（~400行）

需要理解Gumbel分布数学原理